### Architecture

### imports

In [28]:
import os
import pandas as pd
from datetime import datetime

os.makedirs("monitoring", exist_ok=True)

dummy_logs = pd.DataFrame({
    "timestamp": [datetime.now()],
    "prediction": [1],
    "probability": [0.6023]
})

dummy_logs.to_csv(
    "monitoring/prediction_logs.csv",
    index=False
)

print("prediction_logs.csv created.")

prediction_logs.csv created.


### Load User Prediction Logs

In [29]:
prediction_logs = pd.read_csv(
    "monitoring/prediction_logs.csv"
)

prediction_logs.head()

,timestamp,prediction,probability
0,2026-06-24 15:00:44.471660,1,0.6023


### Load Test Dataset

In [30]:
X_test = joblib.load(
    "artifacts/preprocessing/X_test_processed.pkl"
)

y_test = joblib.load(
    "artifacts/preprocessing/y_test.pkl"
)

print(X_test.shape)

(1407, 45)


### Load Production Model

In [31]:
production_model = joblib.load(
    "models/best_model.pkl"
)

print("Production model loaded.")

Production model loaded.


### Performance Monitoring

In [32]:
predictions = production_model.predict(X_test)

probabilities = production_model.predict_proba(X_test)[:,1]

production_metrics = {

    "Accuracy": accuracy_score(y_test, predictions),

    "Precision": precision_score(y_test, predictions),

    "Recall": recall_score(y_test, predictions),

    "F1 Score": f1_score(y_test, predictions),

    "ROC AUC": roc_auc_score(y_test, probabilities)
}

pd.DataFrame(
    production_metrics,
    index=["Production Model"]
)

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Production Model,0.78678,0.612121,0.540107,0.573864,0.82629


###  Define Baseline Metrics

In [33]:
baseline_metrics = {

    "Accuracy": 0.80,

    "F1 Score": 0.72,

    "ROC AUC": 0.84
}

baseline_metrics

{'Accuracy': 0.8, 'F1 Score': 0.72, 'ROC AUC': 0.84}

### Drift Detection

In [34]:
if production_metrics["ROC AUC"] < baseline_metrics["ROC AUC"]:

    retraining_required = True

else:

    retraining_required = False

print("Retraining Required:", retraining_required)

Retraining Required: True


### Retraining Trigger

In [35]:
if retraining_required:

    print("Starting retraining pipeline...")

else:

    print("Current model is healthy.")

Starting retraining pipeline...


###  Load Training Data

In [36]:
X_train = joblib.load(
    "artifacts/preprocessing/X_train_processed.pkl"
)

y_train = joblib.load(
    "artifacts/preprocessing/y_train.pkl"
)

### Train New Model

In [37]:
new_model = LGBMClassifier(
    random_state=42
)

new_model.fit(X_train, y_train)

print("New model trained.")

[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000613 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 668
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
New model trained.


### Evaluate New Model

In [38]:
new_predictions = new_model.predict(X_test)

new_probabilities = new_model.predict_proba(X_test)[:,1]

new_metrics = {

    "Accuracy": accuracy_score(y_test, new_predictions),

    "Precision": precision_score(y_test, new_predictions),

    "Recall": recall_score(y_test, new_predictions),

    "F1 Score": f1_score(y_test, new_predictions),

    "ROC AUC": roc_auc_score(y_test, new_probabilities)
}

pd.DataFrame(
    new_metrics,
    index=["Candidate Model"]
)

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Candidate Model,0.78678,0.612121,0.540107,0.573864,0.82629


### Compare Old vs New

In [39]:
comparison_df = pd.DataFrame({

    "Production Model": production_metrics,

    "Candidate Model": new_metrics

})

comparison_df

,Production Model,Candidate Model
Accuracy,0.786780,0.786780
Precision,0.612121,0.612121
Recall,0.540107,0.540107
F1 Score,0.573864,0.573864
ROC AUC,0.826290,0.826290


### Create Model Registry

In [40]:
os.makedirs(
    "model_registry",
    exist_ok=True
)

###  Save Versioned Model

In [41]:
version = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

version_path = (
    f"model_registry/model_{version}.pkl"
)

joblib.dump(
    new_model,
    version_path
)

print(version_path)

model_registry/model_20260624_150629.pkl


### Promote Best Model to Production

In [42]:
if new_metrics["ROC AUC"] > production_metrics["ROC AUC"]:

    joblib.dump(
        new_model,
        "models/best_model.pkl"
    )

    promoted_model = "New Model"

else:

    promoted_model = "Current Model"

print("Promoted Model:", promoted_model)

Promoted Model: Current Model


### Save Retraining Logs

In [43]:
log_entry = {

    "timestamp": datetime.now(),

    "old_roc_auc": production_metrics["ROC AUC"],

    "new_roc_auc": new_metrics["ROC AUC"],

    "promoted_model": promoted_model
}

log_df = pd.DataFrame([log_entry])

os.makedirs(
    "logs",
    exist_ok=True
)

log_file = "logs/retraining_logs.csv"

if os.path.exists(log_file):

    log_df.to_csv(
        log_file,
        mode="a",
        header=False,
        index=False
    )

else:

    log_df.to_csv(
        log_file,
        index=False
    )

print("Retraining log saved.")

Retraining log saved.


### View Retraining History

In [44]:
pd.read_csv(
    "logs/retraining_logs.csv"
)

,timestamp,production_f1,candidate_f1,best_model
0,2026-06-24 14:53:04.324393,0.573864,0.607955,Logistic Regression
1,2026-06-24 15:07:20.004767,0.826290,0.826290,Current Model
